# Probability for Engineers 
# Chapter 7 — Properties of Expected Value
---

This notebook covers the following topics following a **theory → code → visualization → interpretation** flow:

| # | Topic |
|---|-------|
| 1 | Expected Value of g(X,Y) |
| 2 | Expected Value of a Sum — E[X+Y] = E[X]+E[Y] |
| 3 | Expected Value of a Product — E[XY] and Independence |
| 4 | Covariance and Its Properties |
| 5 | Correlation |
| 6 | Conditional Expected Value and the Law of Total Expectation |
| 7 | Conditional Variance |
| 8 | Moment Generating Functions (MGF) |

In [ ]:
# ── Required libraries ────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from itertools import product as iproduct

np.random.seed(42)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
print("Libraries loaded ✓")

---
## 1 · Expected Value of g(X, Y)

### Theory

Let $g(X,Y)$ be a real-valued function defined on $X$ and $Y$.

$$
E[g(X,Y)] = \begin{cases}
\displaystyle\sum_{x}\sum_{y} g(x,y)\,f(x,y) & \text{discrete case} \\
\displaystyle\int_{-\infty}^{\infty}\int_{-\infty}^{\infty} g(x,y)\,f(x,y)\,dx\,dy & \text{continuous case}
\end{cases}
$$

In [ ]:
# ── Example: Discrete Joint Distribution ──────────────────────────────────────
# X ∈ {1,2,3}, Y ∈ {1,2}  →  joint probability table

x_vals = np.array([1, 2, 3])
y_vals = np.array([1, 2])

# f(x,y) : row=x, column=y
f_xy = np.array([
    [0.10, 0.15],  # x=1
    [0.20, 0.25],  # x=2
    [0.15, 0.15],  # x=3
])

assert np.isclose(f_xy.sum(), 1.0), "Probabilities must sum to 1!"

# Choose g(x,y) = x^2 + y
def g(x, y):
    return x**2 + y

# E[g(X,Y)] = Σ_x Σ_y g(x,y) * f(x,y)
E_g = sum(
    g(x_vals[i], y_vals[j]) * f_xy[i, j]
    for i in range(len(x_vals))
    for j in range(len(y_vals))
)

print("Joint probability table f(x,y):")
print(f"{'':>6}", end="")
for y in y_vals:
    print(f"  Y={y}  ", end="")
print()
for i, x in enumerate(x_vals):
    print(f"X={x}  ", end="")
    for j in range(len(y_vals)):
        print(f"  {f_xy[i,j]:.2f}  ", end="")
    print()

print(f"\nE[g(X,Y)] for g(x,y) = x² + y: {E_g:.4f}")

---
## 2 · Expected Value of a Sum: E[X + Y] = E[X] + E[Y]

### Theory

This property does **not require** $X$ and $Y$ to be independent, and generalizes to $n$ variables:

$$E[X_1 + X_2 + \cdots + X_n] = E[X_1] + E[X_2] + \cdots + E[X_n]$$

**Proof sketch (continuous case):**

$$E(X+Y) = \iint (x+y)f(x,y)\,dx\,dy = \int x f_X(x)\,dx + \int y f_Y(y)\,dy = E(X)+E(Y)$$

In [ ]:
# ── Numerical verification: E[X+Y] = E[X] + E[Y] via Monte Carlo ─────────────
N = 500_000

# X ~ Normal(μ=3, σ=1),  Y ~ Exponential(λ=0.5)  — two non-independent variables
X = np.random.normal(loc=3, scale=1, size=N)
Y = np.random.exponential(scale=2, size=N)   # E[Y] = 2

E_X   = X.mean()
E_Y   = Y.mean()
E_XpY = (X + Y).mean()

print("═" * 45)
print(f"  E[X]          = {E_X:.4f}  (theoretical: 3.0000)")
print(f"  E[Y]          = {E_Y:.4f}  (theoretical: 2.0000)")
print(f"  E[X] + E[Y]   = {E_X + E_Y:.4f}  (theoretical: 5.0000)")
print(f"  E[X + Y]      = {E_XpY:.4f}")
print(f"  Difference    = {abs(E_XpY - (E_X + E_Y)):.6f}")
print("═" * 45)
print("✓ E[X+Y] = E[X]+E[Y]  verified.")

In [ ]:
# ── Visualization: sum of n variables ────────────────────────────────────────
# X_i ~ Bernoulli(p=0.3)  →  S_n = X_1+...+X_n ~ Binomial(n,p)
# E[S_n] = n*p  →  linear growth

p = 0.3
n_values = np.arange(1, 51)
theoretical_E = n_values * p

mc_E = [np.random.binomial(n, p, size=50_000).mean() for n in n_values]

plt.figure(figsize=(8, 4))
plt.plot(n_values, theoretical_E, 'r-', lw=2, label=r'Theoretical: $E[S_n] = np$')
plt.plot(n_values, mc_E, 'b.', ms=4, alpha=0.7, label='Monte Carlo')
plt.xlabel('n (number of variables)')
plt.ylabel(r'$E[S_n]$')
plt.title(r'$E[X_1 + \cdots + X_n] = E[X_1] + \cdots + E[X_n]$  —  Binomial example  ($p=0.3$)')
plt.legend()
plt.tight_layout()
plt.show()

---
## 3 · Expected Value of a Product: E[XY]

### Theory

For **independent** $X$ and $Y$:

$$E[XY] = E[X]\,E[Y]$$

> ⚠️ This formula is valid **only when X and Y are independent** and is **not a sufficient condition** for independence.

In [ ]:
# ── Independent case: E[XY] = E[X]*E[Y] ──────────────────────────────────────
N = 500_000
X_ind = np.random.normal(2, 1, N)
Y_ind = np.random.normal(3, 1, N)   # X ⊥ Y

E_XY_independent = (X_ind * Y_ind).mean()
E_X_E_Y          = X_ind.mean() * Y_ind.mean()

print("── Independent X and Y ──")
print(f"  E[XY]         = {E_XY_independent:.4f}")
print(f"  E[X]·E[Y]     = {E_X_E_Y:.4f}")
print(f"  Equal?         {'✓ Yes' if np.isclose(E_XY_independent, E_X_E_Y, atol=0.02) else '✗ No'}")

# ── Dependent case: Y = 2X + ε ───────────────────────────────────────────────
X_dep = np.random.normal(0, 1, N)
Y_dep = 2 * X_dep + np.random.normal(0, 0.5, N)   # strong dependence

E_XY_dependent  = (X_dep * Y_dep).mean()
E_X_E_Y_dep     = X_dep.mean() * Y_dep.mean()

print("\n── Dependent X and Y  (Y = 2X + ε) ──")
print(f"  E[XY]         = {E_XY_dependent:.4f}")
print(f"  E[X]·E[Y]     = {E_X_E_Y_dep:.4f}")
print(f"  Equal?         {'✓ Yes' if np.isclose(E_XY_dependent, E_X_E_Y_dep, atol=0.05) else '✗ No — dependence present!'}")

---
## 4 · Covariance

### Theory

$$\text{Cov}(X,Y) = E[(X-\mu_X)(Y-\mu_Y)] = E[XY] - \mu_X\mu_Y$$

**Key properties:**

| Property | Expression |
|----------|------------|
| Symmetry | $\text{Cov}(X,Y)=\text{Cov}(Y,X)$ |
| With a constant | $\text{Cov}(X,c)=0$ |
| With itself | $\text{Cov}(X,X)=\text{Var}(X)$ |
| Scaling | $\text{Cov}(aX,bY)=ab\,\text{Cov}(X,Y)$ |
| Translation | $\text{Cov}(X+a,Y+b)=\text{Cov}(X,Y)$ |

**Variance of a sum:**
$$\text{Var}(X+Y) = \text{Var}(X) + \text{Var}(Y) + 2\,\text{Cov}(X,Y)$$
$$\text{Var}(X-Y) = \text{Var}(X) + \text{Var}(Y) - 2\,\text{Cov}(X,Y)$$

In [ ]:
# ── Numerical verification of covariance properties ───────────────────────────
N = 300_000
rng = np.random.default_rng(42)

X = rng.normal(4, 2, N)
Y = rng.normal(1, 3, N) + 0.6 * X   # dependent
a, b, c_const = 2.0, -3.0, 5.0

def cov(A, B):
    return np.mean((A - A.mean()) * (B - B.mean()))

print("Covariance Properties — Numerical Verification")
print("═" * 50)

# Cov(X,Y) = Cov(Y,X)
print(f"Cov(X,Y)           = {cov(X,Y):+.4f}")
print(f"Cov(Y,X)           = {cov(Y,X):+.4f}  ← symmetry ✓")

# Cov(X, constant) = 0
C = np.full(N, c_const)
print(f"Cov(X, {c_const})         = {cov(X,C):+.6f}  ← ≈0 ✓")

# Cov(X,X) = Var(X)
print(f"Cov(X,X)           = {cov(X,X):+.4f}")
print(f"Var(X)             = {X.var():+.4f}  ← equal ✓")

# Cov(aX, bY) = ab * Cov(X,Y)
lhs = cov(a*X, b*Y)
rhs = a * b * cov(X, Y)
print(f"Cov({a}X,{b}Y)       = {lhs:+.4f}")
print(f"ab·Cov(X,Y)        = {rhs:+.4f}  ← scaling ✓")

# Cov(X+a, Y+b) = Cov(X,Y)
lhs2 = cov(X + a, Y + b)
print(f"Cov(X+{a}, Y+{b})   = {lhs2:+.4f}")
print(f"Cov(X,Y)           = {cov(X,Y):+.4f}  ← translation ✓")

# Variance of a sum formula
print("\nVariance of a Sum")
print(f"Var(X+Y) [direct]  = {(X+Y).var():+.4f}")
print(f"Var(X)+Var(Y)+2Cov = {X.var()+Y.var()+2*cov(X,Y):+.4f}  ✓")

In [ ]:
# ── Counterexample: Cov=0 ⟹ NOT Independent ──────────────────────────────────
# Textbook example: X ~ Uniform(-0.5, 0.5),  Y = X²

N = 500_000
X_u = np.random.uniform(-0.5, 0.5, N)
Y_u = X_u**2

cov_val = cov(X_u, Y_u)
E_X3    = np.mean(X_u**3)   # E[XY] = E[X³]

print("Counterexample: X ~ Uniform(-0.5, 0.5),  Y = X²")
print("═" * 48)
print(f"  E[X]           = {X_u.mean():+.6f}  (theoretical: 0)")
print(f"  E[X³] = E[XY]  = {E_X3:+.6f}  (theoretical: 0)")
print(f"  Cov(X,Y)       = {cov_val:+.6f}  (theoretical: 0)")
print()
print("  Since Y = X², Y is completely determined by X.")
print("  Yet Cov(X,Y) ≈ 0  →  Zero covariance does NOT guarantee independence!")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
idx = np.random.choice(N, 3000)

axes[0].scatter(X_u[idx], Y_u[idx], s=5, alpha=0.4, color='steelblue')
axes[0].set_xlabel('X'); axes[0].set_ylabel('Y = X²')
axes[0].set_title(f'Y=X²  |  Cov(X,Y)={cov_val:.5f}\nCov=0 but DEPENDENT!')

X_ind2 = np.random.uniform(-0.5, 0.5, N)
Y_ind2 = np.random.uniform(0, 0.25, N)
axes[1].scatter(X_ind2[idx], Y_ind2[idx], s=5, alpha=0.4, color='coral')
axes[1].set_xlabel('X'); axes[1].set_ylabel('Y')
axes[1].set_title(f'Independent X,Y  |  Cov={cov(X_ind2,Y_ind2):.5f}')

plt.suptitle('Cov=0 ⟹ NOT Independent — Counterexample', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5 · Correlation

### Theory

Covariance depends on the scale of the variables. A scale-free measure is:

$$\rho(X,Y) = \frac{\text{Cov}(X,Y)}{\sqrt{\text{Var}(X)}\,\sqrt{\text{Var}(Y)}}, \quad -1 \leq \rho \leq 1$$

- $|\rho|$ → strength of the linear relationship  
- Sign → positive / negative association  
- $\rho = 0$ → **uncorrelated**

In [ ]:
# ── Visualization of different correlation coefficients ───────────────────────
def generate_sample(rho, n=800, seed=0):
    """Generates a (X,Y) pair with the specified correlation."""
    rng = np.random.default_rng(seed)
    cov_mat = [[1, rho], [rho, 1]]
    return rng.multivariate_normal([0, 0], cov_mat, n).T

rho_values = [1.0, 0.8, 0.4, 0.0, -0.4, -0.8, -1.0]
colors     = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(rho_values)))

fig, axes = plt.subplots(1, len(rho_values), figsize=(16, 3), sharey=True)

for ax, rho, color in zip(axes, rho_values, colors):
    if abs(rho) == 1.0:
        x = np.linspace(-3, 3, 200)
        ax.plot(x, np.sign(rho) * x, color=color, lw=2)
    else:
        x, y = generate_sample(rho)
        ax.scatter(x, y, s=4, alpha=0.5, color=color)
    ax.set_title(f'ρ = {rho:+.1f}', fontsize=10)
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
    ax.set_aspect('equal')
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)

plt.suptitle('Correlation Coefficient — Different Values of ρ', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation computation: verification with NumPy ─────────────────────────
def correlation(X, Y):
    """Pearson correlation coefficient — manual computation."""
    return cov(X, Y) / (X.std() * Y.std())

print("Sample correlation computations:")
print("─" * 40)
for rho_target in [0.9, 0.5, 0.0, -0.5, -0.9]:
    x, y = generate_sample(rho_target, n=100_000)
    rho_hat = correlation(x, y)
    print(f"  Target ρ={rho_target:+.1f}  →  Estimate ρ̂={rho_hat:+.4f}")

---
## 6 · Conditional Expected Value

### Theory

$$E[X \mid Y=y] = \begin{cases}
\displaystyle\sum_x x\, p_{X|Y}(x\mid y) & \text{discrete} \\
\displaystyle\int_{-\infty}^{\infty} x\, f_{X|Y}(x\mid y)\,dx & \text{continuous}
\end{cases}$$

### Law of Total Expectation (Tower Property)

$$\boxed{E[X] = E\bigl[E[X\mid Y]\bigr]}$$

Discrete: $E[X] = \sum_y E[X \mid Y=y]\,P\{Y=y\}$  
Continuous: $E[X] = \int E[X \mid Y=y]\,f_Y(y)\,dy$

In [ ]:
# ── Miner Problem (Textbook Example) ─────────────────────────────────────────
# Door 1: 3 hours → safety
# Door 2: 5 hours → returns to mine
# Door 3: 7 hours → returns to mine
# E[X] = ? (time to reach safety)

# Analytic solution:
# E[X] = (1/3)(3 + (5+E[X]) + (7+E[X]))
# E[X] = (1/3)(15 + 2*E[X])
# 3*E[X] = 15 + 2*E[X]  →  E[X] = 15

def miner_simulation(n_sim=100_000):
    times = []
    for _ in range(n_sim):
        total = 0
        while True:
            door = np.random.choice([1, 2, 3])
            if door == 1:
                total += 3
                break          # reached safety
            elif door == 2:
                total += 5    # returns to mine
            else:
                total += 7    # returns to mine
        times.append(total)
    return np.array(times)

times = miner_simulation()

print("Miner Problem — Expected Value via Conditioning")
print("═" * 48)
print(f"  Analytic solution:  E[X] = 15 hours")
print(f"  Monte Carlo:        E[X] = {times.mean():.3f} hours")
print(f"  Standard deviation: σ    = {times.std():.3f} hours")

# Histogram
plt.figure(figsize=(9, 4))
plt.hist(times, bins=range(0, 200, 3), density=True,
         color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(15, color='red', lw=2, linestyle='--', label='E[X] = 15 (theoretical)')
plt.axvline(times.mean(), color='orange', lw=2, linestyle=':', label=f'MC mean = {times.mean():.2f}')
plt.xlabel('Time to reach safety (hours)')
plt.ylabel('Density')
plt.title('Miner Problem — Simulation Results')
plt.legend()
plt.xlim(0, 150)
plt.tight_layout()
plt.show()

In [ ]:
# ── Numerical verification of the Law of Total Expectation ───────────────────
# Y ~ Poisson(λ=4)  — number of students in class
# X | Y=y ~ Normal(y, 1)  — mean grade

N = 500_000
lam = 4
Y_pois = np.random.poisson(lam, N)          # Y ~ Poisson(4)
X_cond  = np.random.normal(Y_pois, 1)       # X|Y ~ Normal(Y, 1)

# Direct E[X]
E_X_direct = X_cond.mean()

# Tower property: E[E[X|Y]] = Σ_y E[X|Y=y] * P(Y=y)
# E[X|Y=y] = y  (mean of the normal distribution)
# E[E[X|Y]] = E[Y] = λ = 4
E_EXY = np.mean(Y_pois.astype(float))   # since E[X|Y] = Y

print("Law of Total Expectation: E[X] = E[E[X|Y]]")
print("═" * 44)
print(f"  Model: Y~Poisson(4),  X|Y~Normal(Y,1)")
print(f"  Theoretical E[X]    = 4.0000  (= E[Y] = λ)")
print(f"  E[X] direct         = {E_X_direct:.4f}")
print(f"  E[E[X|Y]]           = {E_EXY:.4f}")
print(f"  Equal?              {'✓ Yes' if np.isclose(E_X_direct, E_EXY, atol=0.02) else '✗ No'}")

---
## 7 · Conditional Variance

### Theory

$$\text{Var}(X \mid Y=y) = E\bigl[(X - E[X \mid Y=y])^2 \mid Y=y\bigr]$$

**Law of Total Variance (Eve–Adam Formula):**

$$\boxed{\text{Var}(X) = E[\text{Var}(X\mid Y)] + \text{Var}(E[X\mid Y])}$$

- $E[\text{Var}(X\mid Y)]$ → average of within-group variances  
- $\text{Var}(E[X\mid Y])$ → variance of group means

In [ ]:
# ── Law of Total Variance — Numerical Verification ───────────────────────────
# Class example:
#   Y ~ Uniform{1,2,3}  →  which class
#   X | Y=y ~ Normal(10*y, y)  →  different mean and std per class

N = 1_000_000
Y_class = np.random.choice([1, 2, 3], size=N)
X_grade = np.random.normal(10 * Y_class, Y_class)  # vectorized

# Left term: E[Var(X|Y)]
# Var(X|Y=y) = y² (variance of the normal distribution)
E_var_cond = np.mean(Y_class.astype(float)**2)

# Right term: Var(E[X|Y])
# E[X|Y=y] = 10y  →  Var(10Y) = 100*Var(Y)
E_X_cond   = 10 * Y_class  # E[X|Y] = 10Y
Var_E_cond = np.var(E_X_cond.astype(float))

Var_X_direct = X_grade.var()
Var_X_total  = E_var_cond + Var_E_cond

print("Law of Total Variance: Var(X) = E[Var(X|Y)] + Var(E[X|Y])")
print("═" * 52)
print(f"  Var(X) [direct]              = {Var_X_direct:.4f}")
print(f"  E[Var(X|Y)]  (within-group)  = {E_var_cond:.4f}")
print(f"  Var(E[X|Y])  (between-group) = {Var_E_cond:.4f}")
print(f"  Total                        = {Var_X_total:.4f}")
print(f"  Equal?  {'✓ Yes' if np.isclose(Var_X_direct, Var_X_total, rtol=0.01) else '✗ No'}")

# Visualization: within-group and between-group variance
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
class_color = {1: 'steelblue', 2: 'coral', 3: 'seagreen'}

for cls in [1, 2, 3]:
    mask = Y_class == cls
    axes[cls-1].hist(X_grade[mask], bins=60, density=True,
                     color=class_color[cls], alpha=0.7, edgecolor='white')
    axes[cls-1].axvline(10*cls, color='black', lw=2,
                         linestyle='--', label=f'E[X|Y={cls}]={10*cls}')
    axes[cls-1].set_title(f'Class Y={cls}\n'
                           f'E[X|Y]={10*cls}, Var={cls**2}')
    axes[cls-1].set_xlabel('X (grade)')
    axes[cls-1].legend(fontsize=8)

axes[0].set_ylabel('Density')
plt.suptitle('Law of Total Variance — Within-Group and Between-Group Variance',
             fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8 · Moment Generating Functions (MGF)

### Theory

$$M(t) = E[e^{tX}]$$

**Recovering moments:**

$$M^{(n)}(0) = E[X^n]$$

in particular: $E[X] = M'(0)$ and $\text{Var}(X) = M''(0) - [M'(0)]^2$

**MGF for the Binomial distribution:**

$$M(t) = (pe^t + 1-p)^n \implies E[X]=np,\quad \text{Var}(X)=np(1-p)$$

**Key property:**  For independent $X$ and $Y$: $M_{X+Y}(t) = M_X(t)\cdot M_Y(t)$

In [ ]:
# ── Deriving Binomial mean and variance via MGF ───────────────────────────────
from sympy import symbols, exp, diff, simplify, binomial, Sum, oo, factor

t, p_sym, n_sym = symbols('t p n', real=True, positive=True)

# M(t) = (p*e^t + 1 - p)^n
M = (p_sym * exp(t) + 1 - p_sym)**n_sym

M_prime  = diff(M, t)
M_prime2 = diff(M_prime, t)

E_X_sym   = M_prime.subs(t, 0)
E_X2_sym  = M_prime2.subs(t, 0)
Var_X_sym = simplify(E_X2_sym - E_X_sym**2)

print("Symbolic derivation (SymPy):")
print(f"  M(t)     = (p·eᵗ + 1-p)ⁿ")
print(f"  M'(0)    = E[X]  = {E_X_sym}")
print(f"  Var(X)   = {Var_X_sym}")

In [ ]:
# ── MGF — Comparison for Common Distributions ─────────────────────────────────
t_vals = np.linspace(-2, 2, 400)

def mgf_normal(t, mu=0, sigma=1):
    """X~Normal(μ,σ²): M(t) = exp(μt + σ²t²/2)"""
    return np.exp(mu * t + 0.5 * sigma**2 * t**2)

def mgf_exp(t, lam=1):
    """X~Exp(λ): M(t) = λ/(λ-t), t<λ"""
    safe = np.where(t < lam, t, np.nan)
    return lam / (lam - safe)

def mgf_poisson(t, lam=2):
    """X~Poisson(λ): M(t) = exp(λ(eᵗ-1))"""
    return np.exp(lam * (np.exp(t) - 1))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(t_vals, mgf_normal(t_vals, 1, 1), 'b-', lw=2)
axes[0].plot(t_vals, mgf_normal(t_vals, 0, 2), 'r--', lw=2, label='σ=2')
axes[0].set_ylim(0, 20); axes[0].set_title('Normal MGF\n$M(t)=e^{\\mu t+\\sigma^2 t^2/2}$')
axes[0].set_xlabel('t'); axes[0].legend(['μ=1,σ=1', 'μ=0,σ=2'])

t_exp = t_vals[t_vals < 0.95]
axes[1].plot(t_exp, mgf_exp(t_exp, lam=1), 'g-', lw=2)
axes[1].set_ylim(0, 10); axes[1].set_title('Exponential MGF\n$M(t)=\\lambda/(\\lambda-t), t<\\lambda$')
axes[1].set_xlabel('t'); axes[1].legend(['λ=1'])

axes[2].plot(t_vals, mgf_poisson(t_vals, 2), 'm-', lw=2)
axes[2].plot(t_vals, mgf_poisson(t_vals, 4), 'c--', lw=2)
axes[2].set_ylim(0, 2000); axes[2].set_title('Poisson MGF\n$M(t)=e^{\\lambda(e^t-1)}$')
axes[2].set_xlabel('t'); axes[2].legend(['λ=2', 'λ=4'])

plt.suptitle('Moment Generating Functions for Common Distributions', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── MGF Property 1: MGF of independent sum = product of MGFs ─────────────────
# X ~ Normal(μ₁,σ₁²),  Y ~ Normal(μ₂,σ₂²)  independent
# X+Y ~ Normal(μ₁+μ₂, σ₁²+σ₂²)

mu1, sig1 = 2, 1
mu2, sig2 = 3, 2
t0 = 0.5   # evaluation at t point

M_X            = mgf_normal(t0, mu1, sig1)
M_Y            = mgf_normal(t0, mu2, sig2)
M_XpY_product  = M_X * M_Y
M_XpY_direct   = mgf_normal(t0, mu1+mu2, np.sqrt(sig1**2+sig2**2))

print("MGF Property 1: M_{X+Y}(t) = M_X(t) · M_Y(t)")
print("═" * 48)
print(f"  X ~ Normal({mu1},{sig1}²),  Y ~ Normal({mu2},{sig2}²)")
print(f"  t = {t0}")
print(f"  M_X({t0})              = {M_X:.6f}")
print(f"  M_Y({t0})              = {M_Y:.6f}")
print(f"  M_X · M_Y             = {M_XpY_product:.6f}")
print(f"  M_{{X+Y}}({t0}) [direct] = {M_XpY_direct:.6f}")
print(f"  Equal? {'✓ Yes' if np.isclose(M_XpY_product, M_XpY_direct) else '✗ No'}")

---
## 9 · Summary: All Properties at a Glance

In [ ]:
summary = """
╔══════════════════════════════════════════════════════════════════════════════╗
║        Chapter 7 — Properties of Expected Value: Summary Table             ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Property                │ Formula                                           ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Linearity               │ E[X₁+⋯+Xₙ] = E[X₁]+⋯+E[Xₙ]  (always)         ║
║ Product (independence)  │ E[XY] = E[X]·E[Y]  (if X⊥Y)                    ║
║ Covariance def.         │ Cov(X,Y) = E[XY] − E[X]E[Y]                     ║
║ Variance of sum         │ Var(X+Y) = Var(X)+Var(Y)+2Cov(X,Y)              ║
║ Correlation             │ ρ(X,Y) = Cov(X,Y)/[σ_X · σ_Y] ∈ [−1,1]        ║
║ Independence →          │ Cov(X,Y) = 0  (but converse is NOT true)         ║
║ Law of total expectation│ E[X] = E[E[X|Y]]                                 ║
║ Law of total variance   │ Var(X) = E[Var(X|Y)] + Var(E[X|Y])              ║
║ MGF definition          │ M(t) = E[eᵗˣ]                                    ║
║ MGF moments             │ M⁽ⁿ⁾(0) = E[Xⁿ]                                 ║
║ MGF independent sum     │ M_{X+Y}(t) = M_X(t)·M_Y(t)                      ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
print(summary)

---
## 10 · Exercises

1. For $X \sim \text{Uniform}(0,1)$ and $Y = X^3$, compute $\text{Cov}(X,Y)$ and $\rho(X,Y)$ both analytically and via Monte Carlo.

2. A student takes 10 courses and receives an independent uniform grade between 1 and 5 in each course. Use the MGF to find the expected value and variance of the total score.

3. Generalize the miner problem to 4 doors: door 4 returns the miner in 9 hours. Find $E[X]$ analytically and via simulation.

4. For $X \sim \text{Normal}(0,1)$, show that $M(t)=e^{t^2/2}$ and verify that $M''(0) = E[X^2] = 1$.